# model_sanity_check

In [ ]:
# This cell sets up a reusable environment for testing any CNN version:
# 1. Adds project root to Python path
# 2. Imports torch and model classes from src
# 3. Ensures this notebook works for CNN_V1, V2, V3 without modification

import sys
import os

sys.path.append("..")

import torch
from src.models import CNN_V1

In [ ]:
# This cell diagnoses import and path issues:
# 1. Confirms current working directory
# 2. Confirms Python path
# 3. Attempts safe import of CNN_V1

import os
import sys

print("Current working directory:", os.getcwd())
print("Python path:", sys.path)

sys.path.append("..")

try:
    from src.models import CNN_V1
    model = CNN_V1()
    print("CNN_V1 imported successfully")
    print(model)

except Exception as e:
    print("ERROR DURING IMPORT OR INITIALIZATION:")
    print(e)

Current working directory: d:\ML-PROJECTS\Cifar_10_Computer_Vision_Project_System\notebooks
Python path: ['c:\\Users\\User\\anaconda3\\envs\\cifar10\\python310.zip', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\DLLs', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\lib', 'c:\\Users\\User\\anaconda3\\envs\\cifar10', '', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\lib\\site-packages', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\lib\\site-packages\\win32', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\lib\\site-packages\\win32\\lib', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\lib\\site-packages\\Pythonwin', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..']
CNN_V1 imported successfully
CNN_V1(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()


In [ ]:
# This cell:
# 1. Creates a dummy CIFAR-10 input tensor
# 2. Runs a forward pass through CNN_V1
# 3. Validates output shape matches (batch_size, 10)

x = torch.randn(4, 3, 32, 32)  # batch of 4 fake images

output = model(x)

print("Output shape:", output.shape)  # expected: torch.Size([4, 10])

NameError: name 'model' is not defined

In [ ]:
# This cell:
# 1. Recreates CNN_V1 architecture (CRITICAL STEP)
# 2. Loads trained weights from cnn_v1.pt
# 3. Ensures no architecture mismatch occurs

model = CNN_V1()

state_dict = torch.load("../models/cnn_v1.pt", map_location="cpu")

model.load_state_dict(state_dict)

model.eval()

print("✔ CNN_V1 loaded successfully from frozen .pt file")

✔ CNN_V1 loaded successfully from frozen .pt file


In [ ]:
# This cell:
# 1. Creates dummy CIFAR-10-like input
# 2. Runs forward pass through frozen model
# 3. Validates output shape correctness

x = torch.randn(4, 3, 32, 32)

with torch.no_grad():
    output = model(x)

print("Output shape:", output.shape)
print("Expected shape: [4, 10]")

Output shape: torch.Size([4, 10])
Expected shape: [4, 10]


In [ ]:
# This cell:
# 1. Runs inference on a random input
# 2. Converts logits to class index
# 3. Maps index to CIFAR-10 class name

import torch

classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

x = torch.randn(1, 3, 32, 32)

with torch.no_grad():
    output = model(x)
    predicted_class = torch.argmax(output, dim=1).item()

print("Predicted class index:", predicted_class)
print("Predicted class name:", classes[predicted_class])

Predicted class index: 6
Predicted class name: frog


In [ ]:
# This cell:
# 1. Uses inference module
# 2. Loads frozen CNN_V1 model
# 3. Computes test accuracy from .pt file

from src.inference import run_frozen_model_evaluation
from src.models import CNN_V1

acc = run_frozen_model_evaluation(
    CNN_V1,
    "../models/cnn_v1.pt"
)

print("CNN_V1 Test Accuracy:", round(acc, 4))

CNN_V1 Test Accuracy: 0.7355


In [ ]:
# This cell:
# 1. Uses model registry to instantiate CNN_V1
# 2. Loads frozen weights (cnn_v1.pt)
# 3. Evaluates model using inference module
# 4. Prints final test accuracy

from src.model_registry import get_model
from src.inference import evaluate_model
from src.data_loader import get_cifar10_dataloaders
import torch

# -------------------------
# DEVICE
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# LOAD MODEL VIA REGISTRY
# -------------------------
model = get_model("cnn_v1").to(device)

# -------------------------
# LOAD WEIGHTS
# -------------------------
state_dict = torch.load("../models/cnn_v1.pt", map_location=device)
model.load_state_dict(state_dict)

model.eval()

# -------------------------
# DATA
# -------------------------
_, testloader = get_cifar10_dataloaders(batch_size=64)

# -------------------------
# EVALUATION
# -------------------------
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = correct / total

print("CNN_V1 FINAL TEST ACCURACY:", round(acc, 4))

CNN_V1 FINAL TEST ACCURACY: 0.7355


In [ ]:
# This cell:
# 1. Sets project root path access
# 2. Fixes Python import system for src modules
# 3. Imports required libraries and model definition
# 4. Prepares CIFAR-10 class labels for interpretation

import os
import sys
import torch
import torchvision
import torchvision.transforms as transforms

# Add project root to path so we can import src modules
sys.path.append("..")

from src.models import CNN_V1

# CIFAR-10 class names (IMPORTANT for readable predictions)
classes = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

print("Setup completed successfully")

Setup completed successfully


In [ ]:
# This cell:
# 1. Instantiates CNN_V1 architecture
# 2. Loads frozen weights from models/cnn_v1.pt
# 3. Sets model to evaluation mode (critical for inference)

model = CNN_V1()

model_path = "../models/cnn_v1.pt"
model.load_state_dict(torch.load(model_path, map_location=torch.device("cpu")))

model.eval()

print("CNN_V1 loaded successfully from frozen file")
print(model)

CNN_V1 loaded successfully from frozen file
CNN_V1(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [ ]:
# This cell:
# 1. Loads CIFAR-10 test dataset
# 2. Applies same normalization used in training
# 3. Retrieves one sample image for prediction testing

transform = transforms.Compose([
    transforms.ToTensor(),
])

testset = torchvision.datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=transform
)

# Take one sample image
image, label = testset[0]

# Add batch dimension (model expects batch input)
image = image.unsqueeze(0)

print("Sample loaded")
print("True label:", classes[label])

Sample loaded
True label: cat


In [ ]:
# This cell:
# 1. Runs inference using CNN_V1
# 2. Converts output logits to probabilities
# 3. Extracts predicted class index
# 4. Maps index → class name

with torch.no_grad():
    outputs = model(image)
    _, predicted = torch.max(outputs, 1)

pred_class = classes[predicted.item()]

print("Predicted class index:", predicted.item())
print("Predicted class name:", pred_class)
print("True class name:", classes[label])

Predicted class index: 8
Predicted class name: ship
True class name: cat


In [1]:
import os

print(os.path.abspath("../src/model_registry.py"))

d:\ML-PROJECTS\Cifar_10_Computer_Vision_Project_System\src\model_registry.py


In [2]:
import sys
import importlib

sys.path.append("..")

import src.model_registry
importlib.reload(src.model_registry)

from src.model_registry import get_model

In [3]:
model = get_model("cnn_v2")

In [4]:
"""
========================================
CNN_V2 FROZEN MODEL TEST (SAFE VERSION)
========================================

This cell:
1. Loads CNN_V2 architecture directly
2. Loads cnn_v2.pt weights
3. Runs in evaluation mode
4. Avoids registry/cache/import issues
"""

import os
import sys
import torch

# Ensure project root is accessible
sys.path.append("..")

from src.models import CNN_V2  # direct architecture import

# -----------------------------
# 1. INSTANTIATE MODEL
# -----------------------------
model = CNN_V2()

# -----------------------------
# 2. LOAD WEIGHTS
# -----------------------------
model_path = "../models/cnn_v2.pt"

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found: {model_path}")

state_dict = torch.load(model_path, map_location=torch.device("cpu"))
model.load_state_dict(state_dict)

# -----------------------------
# 3. SET EVAL MODE
# -----------------------------
model.eval()

print("✅ CNN_V2 loaded successfully from frozen .pt file")
print(model)

✅ CNN_V2 loaded successfully from frozen .pt file
CNN_V2(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_featur

In [7]:
"""
========================================
CNN_V2 SINGLE IMAGE INFERENCE (FIXED)
========================================
"""

import torch
import torchvision
import torchvision.transforms as transforms

# CIFAR-10 class names
classes = ['plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

# -----------------------------
# SAME TRANSFORM AS TRAINING (IMPORTANT FIX)
# -----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# -----------------------------
# LOAD DATASET
# -----------------------------
testset = torchvision.datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=transform
)

# Pick one image
image, label = testset[0]

# Add batch dimension
input_image = image.unsqueeze(0)

# -----------------------------
# PREDICTION
# -----------------------------
model.eval()

with torch.no_grad():
    outputs = model(input_image)
    _, predicted = torch.max(outputs, 1)

print("Predicted class index:", predicted.item())
print("Predicted class name:", classes[predicted.item()])
print("True class name:", classes[label])

Predicted class index: 3
Predicted class name: cat
True class name: cat


In [8]:
"""
========================================
CNN_V2 FULL TEST ACCURACY (FIXED)
========================================
"""

import torch
import torchvision
import torchvision.transforms as transforms

# -----------------------------
# SAME TRANSFORM AS TRAINING
# -----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# -----------------------------
# LOAD TEST DATA
# -----------------------------
testset = torchvision.datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=transform
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)

# -----------------------------
# EVALUATION
# -----------------------------
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total

print("Test Accuracy:", round(accuracy, 4))

Test Accuracy: 0.5263


In [10]:
# This cell:
# 1. Uses model registry to instantiate CNN_V2
# 2. Loads frozen weights (cnn_v1.pt)
# 3. Evaluates model using inference module
# 4. Prints final test accuracy

from src.model_registry import get_model
from src.inference import evaluate_model
from src.data_loader import get_cifar10_dataloaders
import torch

# -------------------------
# DEVICE
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# LOAD MODEL VIA REGISTRY
# -------------------------
model = get_model("cnn_v2").to(device)

# -------------------------
# LOAD WEIGHTS
# -------------------------
state_dict = torch.load("../models/cnn_v2.pt", map_location=device)
model.load_state_dict(state_dict)

model.eval()

# -------------------------
# DATA
# -------------------------
_, testloader = get_cifar10_dataloaders(batch_size=64)

# -------------------------
# EVALUATION
# -------------------------
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = correct / total

print("CNN_V2 FINAL TEST ACCURACY:", round(acc, 4))

CNN_V2 FINAL TEST ACCURACY: 0.7546


In [6]:
from src.model_registry import get_model

print(get_model)

<function get_model at 0x0000019164E4B370>


In [7]:
"""
========================================
CNN_V3 LOAD + SANITY CHECK
========================================
"""

import torch
from src.model_registry import get_model
from src.data_loader import get_cifar10_dataloaders

# -------------------------
# DEVICE
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# LOAD MODEL (FROM REGISTRY)
# -------------------------
model = get_model("cnn_v3").to(device)

# -------------------------
# LOAD WEIGHTS
# -------------------------
model_path = "../models/cnn_v3.pt"
model.load_state_dict(torch.load(model_path, map_location=device))

model.eval()

print("CNN_V3 loaded successfully")
print(model)

CNN_V3 loaded successfully
CNN_V3(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [ ]:
"""
========================================
CNN_V3 TEST ACCURACY
========================================
"""

_, testloader = get_cifar10_dataloaders(batch_size=64)

correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = correct / total

print("CNN_V3 FINAL TEST ACCURACY:", round(acc, 4))

CNN_V3 FINAL TEST ACCURACY: 0.7504


In [ ]:
model = get_model("cnn_v4").to(device)
model.load_state_dict(torch.load("../models/cnn_v4.pt", map_location=device))
model.eval()

_, testloader = get_cifar10_dataloaders(batch_size=64)

correct, total = 0, 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

print("CNN_V4 FINAL TEST ACCURACY:", correct / total)

ValueError: Model cnn_v4 not found in registry